## tl;dr

This outcome-blind diagnostic rejects paired-book **depth symmetry** as a new strategy mechanism. Across `490` valid paired states, all `490` pass a fixed `0.25` matched-depth ratio; the median ratio is exactly `1.0` and the minimum is `0.845`. Complement-side depth is therefore structurally mirrored, not an independent uncertainty signal.

A `$10` both-entry capacity check passes `470 / 490`, but every failure is on the YES-side book. It adds `0` rejections when YES is the chosen outcome and `20` when NO is chosen, exposing outcome-label/price asymmetry rather than a direction-neutral pair feature. The four-side version passes `460 / 490` and is stable across five snapshots, but it is a generic liquidity veto already substantially represented by chosen-book depth and executable FOK checks.

No depth challenger is preregistered. This notebook uses no settlement labels, strategy outcomes, active forward block data, or PnL; no current rule, threshold, grade, collector, or runtime behavior changes.

## Context & Methods

### Key Assumptions

- The compressed public snapshot is authoritative for this bounded structural diagnostic and is pinned by SHA-256.
- REST book arrays are not assumed to be best-first; best prices and the nearest three levels are derived by numeric extrema.
- The planned maximum per-market exposure is `$10`, so `$10` of top-three notional capacity is evaluated as an economic reference point rather than an outcome-fitted threshold.
- YES-bid depth is economically paired with NO-ask depth, and YES-ask depth with NO-bid depth. Their ratios describe complement-side depth symmetry.
- Structural coverage, persistence, or distinctness cannot establish settlement prediction or profitability.

In [1]:
from pathlib import Path
import gzip
import hashlib
import json
import math
import os
import statistics
import tempfile

ROOT = Path('/Users/ttoomm/Documents/PolyMomentum')
SOURCE = ROOT / 'deploy/promotions/evidence/strategy_registry/source_snapshots/20260721_non_candle_public_books.json.gz'
MANIFEST = ROOT / 'deploy/promotions/evidence/strategy_registry/source_snapshots/20260721_non_candle_public_books_manifest.json'
RESIDUAL = ROOT / 'deploy/promotions/evidence/strategy_registry/20260721_binary_complement_residual_cross_market_replication.json'
OUTPUT = ROOT / 'deploy/promotions/evidence/strategy_registry/20260721_binary_complement_paired_depth_capacity_diagnostic.json'

source_sha256 = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
manifest = json.loads(MANIFEST.read_text())
residual = json.loads(RESIDUAL.read_text())
with gzip.open(SOURCE, 'rt') as handle:
    raw = json.load(handle)

assert source_sha256 == manifest['snapshot_sha256']
assert len(raw['markets']) == 100
assert len(raw['rounds']) == 5
assert all(row['response_books'] == 200 for row in raw['rounds'])
print({'source_sha256': source_sha256, 'markets': len(raw['markets']), 'rounds': len(raw['rounds'])})

{'source_sha256': '5fd25a3e78e1f46c556b1adcb6e05d0939f5fd475d4ddf48136863588fc78811', 'markets': 100, 'rounds': 5}


## Data

### 1. Reconstruct valid paired top-three books

In [2]:
def numeric_levels(levels):
    parsed = []
    for level in levels or []:
        try:
            price = float(level['price'])
            size = float(level['size'])
        except (KeyError, TypeError, ValueError):
            continue
        if math.isfinite(price) and math.isfinite(size) and 0 < price < 1 and size > 0:
            parsed.append((price, size))
    return parsed


def book_features(book):
    bids = sorted(numeric_levels(book.get('bids')), key=lambda row: row[0], reverse=True)
    asks = sorted(numeric_levels(book.get('asks')), key=lambda row: row[0])
    if not bids or not asks or bids[0][0] >= asks[0][0]:
        return None
    top_bids = bids[:3]
    top_asks = asks[:3]
    bid_depth = sum(size for _, size in top_bids)
    ask_depth = sum(size for _, size in top_asks)
    best_bid = top_bids[0][0]
    best_ask = top_asks[0][0]
    return {
        'best_bid': best_bid,
        'best_ask': best_ask,
        'bid_depth': bid_depth,
        'ask_depth': ask_depth,
        'bid_notional': sum(price * size for price, size in top_bids),
        'ask_notional': sum(price * size for price, size in top_asks),
        'timestamp_ms': int(book['timestamp']),
        'declared_tick': float(book['tick_size']),
    }


markets = {row['condition_id']: row for row in raw['markets']}
token_to_condition = {
    token: condition_id
    for condition_id, market in markets.items()
    for token in market['token_ids']
}
states = []
missing_or_invalid = []
for round_row in raw['rounds']:
    by_token = {str(book['asset_id']): book for book in round_row['books']}
    for condition_id, market in markets.items():
        pair = []
        for token in market['token_ids']:
            book = by_token.get(token)
            pair.append(book_features(book) if book else None)
        if pair[0] is None or pair[1] is None:
            missing_or_invalid.append((round_row['round_index'], condition_id))
            continue
        yes, no = pair
        yes_mid = (yes['best_bid'] + yes['best_ask']) / 2
        no_mid = (no['best_bid'] + no['best_ask']) / 2
        yes_micro = (
            yes['best_ask'] * yes['bid_depth'] + yes['best_bid'] * yes['ask_depth']
        ) / (yes['bid_depth'] + yes['ask_depth'])
        no_micro = (
            no['best_ask'] * no['bid_depth'] + no['best_bid'] * no['ask_depth']
        ) / (no['bid_depth'] + no['ask_depth'])
        band_tick = 0.001 if any(
            0 < price < 0.04 or price > 0.96
            for price in (yes['best_bid'], yes['best_ask'], no['best_bid'], no['best_ask'])
        ) else 0.01
        effective_tick = min(max(yes['declared_tick'], no['declared_tick']), band_tick)
        matched_ratios = [
            min(yes['bid_depth'], no['ask_depth']) / max(yes['bid_depth'], no['ask_depth']),
            min(yes['ask_depth'], no['bid_depth']) / max(yes['ask_depth'], no['bid_depth']),
        ]
        states.append({
            'round': round_row['round_index'],
            'condition_id': condition_id,
            'question': market['question'],
            'timestamp_skew_ms': abs(yes['timestamp_ms'] - no['timestamp_ms']),
            'mid_residual': abs(yes_mid + no_mid - 1),
            'micro_residual': abs(yes_micro + no_micro - 1),
            'effective_tick': effective_tick,
            'passes_registered_residual': max(abs(yes_mid + no_mid - 1), abs(yes_micro + no_micro - 1)) <= 2 * effective_tick + 1e-12,
            'min_side_depth_shares': min(yes['bid_depth'], yes['ask_depth'], no['bid_depth'], no['ask_depth']),
            'min_entry_ask_capacity_usd': min(yes['ask_notional'], no['ask_notional']),
            'yes_entry_ask_capacity_usd': yes['ask_notional'],
            'no_entry_ask_capacity_usd': no['ask_notional'],
            'min_exit_bid_capacity_usd': min(yes['bid_notional'], no['bid_notional']),
            'min_four_side_capacity_usd': min(yes['bid_notional'], yes['ask_notional'], no['bid_notional'], no['ask_notional']),
            'min_matched_depth_ratio': min(matched_ratios),
            'max_matched_depth_multiple': 1 / min(matched_ratios),
            'passes_entry_capacity_10usd': min(yes['ask_notional'], no['ask_notional']) >= 10,
            'yes_passes_entry_capacity_10usd': yes['ask_notional'] >= 10,
            'no_passes_entry_capacity_10usd': no['ask_notional'] >= 10,
            'passes_four_side_capacity_10usd': min(yes['bid_notional'], yes['ask_notional'], no['bid_notional'], no['ask_notional']) >= 10,
            'passes_matched_depth_ratio_0_25': min(matched_ratios) >= 0.25,
        })

assert len(states) == residual['structural_results']['valid_paired_states'] == 490
assert len(missing_or_invalid) == 10
assert sum(row['passes_registered_residual'] for row in states) == residual['structural_results']['registered_fixed_rule_passes'] == 490
print({'valid_states': len(states), 'missing_or_invalid': len(missing_or_invalid)})

{'valid_states': 490, 'missing_or_invalid': 10}


## Results

### 2. Quantify capacity, symmetry, and incremental selectivity

In [3]:
def quantiles(values):
    ordered = sorted(values)
    def q(p):
        index = (len(ordered) - 1) * p
        lower = math.floor(index)
        upper = math.ceil(index)
        if lower == upper:
            return ordered[lower]
        return ordered[lower] + (ordered[upper] - ordered[lower]) * (index - lower)
    return {'min': ordered[0], 'p10': q(0.10), 'p50': q(0.50), 'p90': q(0.90), 'max': ordered[-1]}


coverage = {
    'entry_capacity_10usd': sum(row['passes_entry_capacity_10usd'] for row in states),
    'yes_entry_capacity_10usd': sum(row['yes_passes_entry_capacity_10usd'] for row in states),
    'no_entry_capacity_10usd': sum(row['no_passes_entry_capacity_10usd'] for row in states),
    'four_side_capacity_10usd': sum(row['passes_four_side_capacity_10usd'] for row in states),
    'matched_depth_ratio_0_25': sum(row['passes_matched_depth_ratio_0_25'] for row in states),
}
coverage_rates = {key: value / len(states) for key, value in coverage.items()}
disagreement = {
    'four_side_vs_residual': sum(
        row['passes_four_side_capacity_10usd'] != row['passes_registered_residual']
        for row in states
    ),
    'depth_ratio_vs_residual_nonzero': sum(
        (not row['passes_matched_depth_ratio_0_25']) and row['passes_registered_residual']
        for row in states
    ),
    'opposite_capacity_incremental_if_yes_chosen': sum(
        row['yes_passes_entry_capacity_10usd'] and not row['passes_entry_capacity_10usd']
        for row in states
    ),
    'opposite_capacity_incremental_if_no_chosen': sum(
        row['no_passes_entry_capacity_10usd'] and not row['passes_entry_capacity_10usd']
        for row in states
    ),
}
summary = {
    'coverage_counts': coverage,
    'coverage_rates': coverage_rates,
    'min_entry_ask_capacity_usd': quantiles([row['min_entry_ask_capacity_usd'] for row in states]),
    'min_exit_bid_capacity_usd': quantiles([row['min_exit_bid_capacity_usd'] for row in states]),
    'min_four_side_capacity_usd': quantiles([row['min_four_side_capacity_usd'] for row in states]),
    'min_matched_depth_ratio': quantiles([row['min_matched_depth_ratio'] for row in states]),
    'timestamp_skew_ms': quantiles([row['timestamp_skew_ms'] for row in states]),
    'disagreement': disagreement,
}
print(json.dumps(summary, indent=2))

{
  "coverage_counts": {
    "entry_capacity_10usd": 470,
    "yes_entry_capacity_10usd": 470,
    "no_entry_capacity_10usd": 490,
    "four_side_capacity_10usd": 460,
    "matched_depth_ratio_0_25": 490
  },
  "coverage_rates": {
    "entry_capacity_10usd": 0.9591836734693877,
    "yes_entry_capacity_10usd": 0.9591836734693877,
    "no_entry_capacity_10usd": 1.0,
    "four_side_capacity_10usd": 0.9387755102040817,
    "matched_depth_ratio_0_25": 1.0
  },
  "min_entry_ask_capacity_usd": {
    "min": 1.85156,
    "p10": 15.96425,
    "p50": 510.19731,
    "p90": 7853.499420000001,
    "max": 65889.9539
  },
  "min_exit_bid_capacity_usd": {
    "min": 2.9200000000000004,
    "p10": 22.77736,
    "p50": 599.316605,
    "p90": 12840.7203,
    "max": 122560.4766
  },
  "min_four_side_capacity_usd": {
    "min": 1.85156,
    "p10": 12.4779,
    "p50": 401.85685,
    "p90": 7577.75774,
    "max": 65889.9539
  },
  "min_matched_depth_ratio": {
    "min": 0.8451908165966944,
    "p10": 1.0,
   

### 3. Check five-round stability at the market level

In [4]:
by_market = {}
for row in states:
    by_market.setdefault(row['condition_id'], []).append(row)

market_stability = []
for condition_id, rows in sorted(by_market.items()):
    if len(rows) != 5:
        continue
    passes = [row['passes_four_side_capacity_10usd'] for row in rows]
    ratios = [row['min_matched_depth_ratio'] for row in rows]
    market_stability.append({
        'condition_id': condition_id,
        'pass_count': sum(passes),
        'flips': sum(left != right for left, right in zip(passes, passes[1:])),
        'ratio_range': max(ratios) - min(ratios),
    })

stability = {
    'complete_five_round_markets': len(market_stability),
    'always_pass': sum(row['pass_count'] == 5 for row in market_stability),
    'always_fail': sum(row['pass_count'] == 0 for row in market_stability),
    'mixed': sum(0 < row['pass_count'] < 5 for row in market_stability),
    'markets_with_any_flip': sum(row['flips'] > 0 for row in market_stability),
    'maximum_flips': max((row['flips'] for row in market_stability), default=0),
    'matched_ratio_range': quantiles([row['ratio_range'] for row in market_stability]),
}
print(json.dumps(stability, indent=2))

{
  "complete_five_round_markets": 98,
  "always_pass": 92,
  "always_fail": 6,
  "mixed": 0,
  "markets_with_any_flip": 0,
  "maximum_flips": 0,
  "matched_ratio_range": {
    "min": 0.0,
    "p10": 0.0,
    "p50": 0.0,
    "p90": 0.0,
    "max": 0.0006453427965005254
  }
}


### 4. Export the source-pinned diagnostic

In [5]:
artifact = {
    'schema_version': 1,
    'generated_at': '2026-07-21T07:11:33Z',
    'status': 'LABEL_FREE_PAIRED_DEPTH_CHALLENGER_REJECTED_AS_NON_DISTINCT',
    'decision_question': 'Does paired-book depth capacity add stable structural selectivity beyond the observed-inert complement residual?',
    'source_authority': {
        'snapshot': str(SOURCE.relative_to(ROOT)),
        'snapshot_sha256': source_sha256,
        'manifest': str(MANIFEST.relative_to(ROOT)),
        'terminal_labels_loaded': False,
        'strategy_outcomes_loaded': False,
        'active_forward_block_loaded': False,
    },
    'population': {
        'markets': len(markets),
        'rounds': len(raw['rounds']),
        'possible_paired_states': len(markets) * len(raw['rounds']),
        'valid_paired_states': len(states),
        'missing_or_invalid_states': len(missing_or_invalid),
        'definition': 'top 100 active, accepting, non-crypto, non-negative-risk binary CLOB markets by volume24hr after deterministic filtering of 500 Gamma rows',
    },
    'methodology': {
        'top_book_policy': 'derive best bid by maximum price, best ask by minimum price, and depth from the nearest three valid levels',
        'economic_reference': '$10 top-three notional equals the standing maximum per-market exposure; diagnostic only, not outcome-fitted',
        'matched_depth_pairs': ['YES bid vs NO ask', 'YES ask vs NO bid'],
        'candidate_fields': ['minimum four-side top-three notional capacity', 'minimum matched-depth ratio'],
    },
    'structural_results': summary,
    'five_round_stability': stability,
    'mechanism_assessment': {
        'distinct_pair_signal_observed': False,
        'matched_depth_interpretation': 'Complement-side top-three share depth is effectively mirrored; the fixed 0.25 symmetry rule is inert in all 490 valid states.',
        'capacity_interpretation': 'Dollar-notional capacity rejects some states but is outcome-label and price asymmetric rather than a direction-neutral paired-book signal.',
        'existing_control_overlap': 'Chosen-book minimum depth and executable FOK capacity already control the directly actionable liquidity boundary.',
    },
    'decision': {
        'strategy_rule_changed': False,
        'threshold_registered': False,
        'paired_depth_challenger_preregistered': False,
        'profitability_claim': False,
        'a_plus_claim': False,
        'next_step': 'Do not register or implement a paired-depth challenger; preserve the sealed primary block and rely on schema-6 attribution to test paired-book validity without multiplying hypotheses.',
    },
    'limitations': [
        'Five snapshots over roughly nine seconds cannot establish longer-horizon persistence.',
        'The population is non-candle and may not match five-minute BTC market liquidity.',
        'No terminal labels, historical trade rows, fills, latency, or PnL are present.',
    ],
}

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
fd, temp_path = tempfile.mkstemp(prefix=OUTPUT.name + '.tmp.', dir=OUTPUT.parent)
try:
    with os.fdopen(fd, 'w') as handle:
        json.dump(artifact, handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temp_path, OUTPUT)
finally:
    if os.path.exists(temp_path):
        os.unlink(temp_path)

print({'artifact': str(OUTPUT.relative_to(ROOT)), 'sha256': hashlib.sha256(OUTPUT.read_bytes()).hexdigest()})

{'artifact': 'deploy/promotions/evidence/strategy_registry/20260721_binary_complement_paired_depth_capacity_diagnostic.json', 'sha256': 'b3bb59fb27805c72be7263176161d33fafe3536775ba311a9ccb636acd7af7b5'}


## Takeaways

- Matched paired depth is inert: `490 / 490` states pass the fixed symmetry reference and the median matched ratio is exactly `1.0`.
- Dollar capacity is not direction-neutral: all `20` both-entry failures occur on the YES book, producing `0` incremental rejections for YES-chosen rows and `20` for NO-chosen rows.
- Do **not** preregister or implement this challenger. It adds hypothesis multiplicity without demonstrated pair-specific information.
- The existing binary-complement rule, schema-6 attribution, paper-only grade, and active forward collection remain unchanged.